### LASSO solo con variables EPH — bielección (bloque largo t-2 -> t)

Misma estructura que `ventana_t-1/04_lasso_eph_local.ipynb`: las 6
variables EPH-derivadas (D21) contra los 4 targets del panel, sobre
`data/tfi_data/panel_ventanas_bieleccion.csv` con features de trayectoria
trimestral `_trim` en vez de `_vc`.

In [ ]:
import sys
import pandas as pd
import numpy as np

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")
from ml_models.cargar_panel import cargar_panel, columnas_candidatas
from ml_models.lasso import *

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel, f"{data_path}panel_ventanas_bieleccion.csv") for nivel in NIVELES}
for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

In [ ]:
PREFIJOS_EPH = [
    "tasa_informalidad", "pct_sin_cobertura_salud", "hacinamiento_medio",
    "pct_hogares_ayuda_social_gobierno", "pct_hogares_prestamo_bancario",
    "pct_hogares_vendio_pertenencias",
]
TARGETS = ["delta_v", "delta_participacion_pct", "delta_voto_exit_total_pct", "magnitud_desplazamiento_ideologico"]

# delta_voto_exit_total_pct no es columna del panel (D22) -- se reconstruye
# igual que en 01.3_lasso_voto_exit.ipynb.
for nivel in NIVELES:
    df = paneles[nivel]
    if "delta_voto_exit_total_pct" not in df.columns:
        df["delta_voto_exit_total_pct"] = df["delta_voto_exit_ausentismo_pct"] + df["delta_voto_exit_blanco_nulo_pct"]

In [ ]:
cols_eph_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols = columnas_candidatas(df, excluir_adicional=TARGETS + ["delta_voto_exit_ausentismo_pct", "delta_voto_exit_blanco_nulo_pct"])
    cols_eph_por_nivel[nivel] = [c for c in cols if any(c.startswith(p) for p in PREFIJOS_EPH)]
    print(f"{nivel}: {len(cols_eph_por_nivel[nivel])} columnas EPH candidatas, N={len(df)}")

In [ ]:
UMBRAL_REDUNDANCIA = 0.90
ORDEN_SUFIJO = ["_nivel_trim", "_delta_nivel", "_final_trim", "_pendiente_trim", "_volatilidad_trim", "_cobertura_parcial"]
cols_finales_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols_eph = cols_eph_por_nivel[nivel]
    corr = df[cols_eph].corr(method="pearson")
    clusters = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    representantes = [elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO) for c in clusters]
    cols_finales_por_nivel[nivel] = sorted(representantes)
    print(f"{nivel}: {len(cols_eph)} -> {len(representantes)} tras colapsar clusters (>= |{UMBRAL_REDUNDANCIA}|)")
    for c in clusters:
        if len(c) > 1:
            rep = elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO)
            print(f"    cluster: {sorted(c)} -> representante: {rep}")

In [ ]:
resumen_filas = []
for target in TARGETS:
    print(f"\n{'#'*70}\nOBJETIVO: {target}\n{'#'*70}")
    for nivel in NIVELES:
        df = paneles[nivel]
        cols = cols_finales_por_nivel[nivel]
        X, y = construir_Xy_final(nivel, cols, paneles, target=target)
        n, p = X.shape
        if n < 4:
            print(f"  [{nivel}] N={n} insuficiente para LOO-CV, se salta")
            continue
        baseline = baseline_trivial_loocv(y)
        resultado_cv = lasso_loocv_manual(X, y, n_alphas=50)
        mse_min = mse_en_alpha(X, y, resultado_cv["alpha_min"])
        mejora = 100 * (1 - mse_min / baseline)
        coef = ajustar_final(X, y, resultado_cv["alpha_min"])
        coef_no_cero = coef[coef != 0]
        print(f"  [{nivel}] N={n}, P={p}, mejora_alpha_min={mejora:.2f}%, coef≠0: {dict(coef_no_cero.round(3))}")
        resumen_filas.append({
            "target": target, "nivel": nivel, "N": n, "P": p,
            "mejora_alpha_min_%": mejora, "n_coef_no_cero": len(coef_no_cero),
        })

tabla_resumen = pd.DataFrame(resumen_filas)
tabla_resumen